# Analisis de Finanzas y Riesgo Crediticio

Proyecto: Banca Atlas

Equip_30

Equipo: Finanzas y Riesgo Crediticio 

Mariia Zaitseva

_____

Semana: 2

_____

## Pregunta de negocio
Los clientes con préstamos e hipotecas tienden a tener un
¿saldo medio más bajo o más riesgo de incumplimiento? Cómo deberíamos ajustar nuestras ofertas y
estrategias de gestión de riesgos en función de estos hallazgos?

### Imports

In [3]:
import pandas as pd
import numpy as np
import scipy.stats as stats

# graficos
import plotly.io as pio
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# DarkMode Plotly
pio.templates.default = "plotly_dark"

pio.templates["custom"] = pio.templates["plotly_dark"]
pio.templates["custom"].layout.paper_bgcolor = "#050a30"
pio.templates["custom"].layout.plot_bgcolor  = "#050a30"
pio.templates.default = "custom"

### Carga de datos

In [4]:
df = pd.read_csv("../../Data/06-01-2026/06-01-2026_Clean.csv", index_col=False, encoding='utf-8')

In [5]:
df

,id,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,deposit
0,1,59,admin.,married,secondary,0,2343,1,0,unknown,5,may,1042,1,-1,0,no_pcampaign,1
1,2,56,admin.,married,secondary,0,45,0,0,unknown,5,may,1467,1,-1,0,no_pcampaign,1
2,3,41,technician,married,secondary,0,1270,1,0,unknown,5,may,1389,1,-1,0,no_pcampaign,1
3,4,55,services,married,secondary,0,2476,1,0,unknown,5,may,579,1,-1,0,no_pcampaign,1
4,5,54,admin.,married,tertiary,0,184,0,0,unknown,5,may,673,2,-1,0,no_pcampaign,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10982,10983,40,management,married,secondary,0,8486,0,0,unknown,6,may,260,3,-1,0,no_pcampaign,0
10983,10984,53,management,married,tertiary,0,20772,0,0,cellular,4,feb,715,1,-1,0,no_pcampaign,0
10984,10985,55,blue-collar,married,primary,0,3297,1,1,telephone,30,apr,96,1,-1,0,no_pcampaign,0
10985,10986,41,management,married,tertiary,0,9,1,0,cellular,22,jul,82,3,-1,0,no_pcampaign,0


In [6]:
df_clients = pd.read_csv("../../Data/06-01-2026/06-01-2026_Clusters_clients.csv", encoding='utf-8')

In [7]:
df_clients

,Unnamed: 0,age,job,marital,education,age_scaled,cluster,deposit,loan,housing,total_products_bought
0,0,59,admin.,married,secondary,1.485350,0,1,0,1,2
1,1,56,admin.,married,secondary,1.234172,0,1,0,0,1
2,2,41,technician,married,secondary,-0.021718,1,1,0,1,2
3,3,55,services,married,secondary,1.150446,0,1,0,1,2
4,4,54,admin.,married,tertiary,1.066720,0,1,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...
10982,10982,40,management,married,secondary,-0.105444,1,0,0,0,0
10983,10983,53,management,married,tertiary,0.982994,0,0,0,0,0
10984,10984,55,blue-collar,married,primary,1.150446,0,0,1,1,2
10985,10985,41,management,married,tertiary,-0.021718,2,0,0,1,1


In [8]:
df = df.merge(df_clients['cluster'], left_index=True, right_index=True)

In [9]:
df

,id,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,deposit,cluster
0,1,59,admin.,married,secondary,0,2343,1,0,unknown,5,may,1042,1,-1,0,no_pcampaign,1,0
1,2,56,admin.,married,secondary,0,45,0,0,unknown,5,may,1467,1,-1,0,no_pcampaign,1,0
2,3,41,technician,married,secondary,0,1270,1,0,unknown,5,may,1389,1,-1,0,no_pcampaign,1,1
3,4,55,services,married,secondary,0,2476,1,0,unknown,5,may,579,1,-1,0,no_pcampaign,1,0
4,5,54,admin.,married,tertiary,0,184,0,0,unknown,5,may,673,2,-1,0,no_pcampaign,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10982,10983,40,management,married,secondary,0,8486,0,0,unknown,6,may,260,3,-1,0,no_pcampaign,0,1
10983,10984,53,management,married,tertiary,0,20772,0,0,cellular,4,feb,715,1,-1,0,no_pcampaign,0,0
10984,10985,55,blue-collar,married,primary,0,3297,1,1,telephone,30,apr,96,1,-1,0,no_pcampaign,0,0
10985,10986,41,management,married,tertiary,0,9,1,0,cellular,22,jul,82,3,-1,0,no_pcampaign,0,2


### Exploración de los Datos para el analisis

#### Matrix de correlación de variables:

Crear dataframe

In [10]:
cols_to_corr = [
    'age',
    'job',
    'marital',
    'education',
    'default',
    'balance',
    'housing',
    'loan',
    'cluster',
]
df_subset = df[cols_to_corr].copy()

In [11]:
education_map = {"primary": 1, "secondary": 2, "tertiary": 3, "unknown": 0}
df_subset["education"] = df_subset["education"].map(education_map).fillna(0)

for col in ["job", "marital", "default", "housing", "loan"]:
    df_subset[col] = df_subset[col].factorize()[0]

In [12]:
corr_matrix = df_subset.corr(method="spearman")

In [13]:
fig = px.imshow(
    corr_matrix,
    text_auto=".2f",  
    aspect="auto",
    color_continuous_scale="RdBu_r", 
    zmin=-1,
    zmax=1, 
    labels=dict(color="<b>Correlación<br>de Spearman</b>"),
    title="<b>Matrix de correlación (Correlación de rangos de Spearman)</b>",
)

fig.update_layout(
    width=800,
    height=700,
    title_font={"size": 16},
    xaxis_tickangle=-45,  
)

fig.show()

**Conclusión**:

La correlación entre las variables financias y sociodemográficas es muy debil.

##### El porcentaje de impagos:

Hay pocos casos de pagos atrasados de un préstamo.

In [14]:
df['default'].value_counts(normalize=True) * 100

default
0    98.534632
1     1.465368
Name: proportion, dtype: float64

In [15]:
df_plot = df.copy()
df_plot['default_label'] = df_plot['default'].map({0: 'No', 1: 'Sí'})

fig = px.histogram(
    df_plot, 
    x='default_label', 
    title='La distribution de impagos',
    labels={'default_label': 'Impago (default)', 'count': 'Número de clientes'},
    category_orders={'default_label': ['No', 'Sí']},
    color='default_label',
    color_discrete_sequence=['#636EFA', '#EF553B']
)

fig.update_layout(
    title_font=dict(size=16, weight='bold'),
    xaxis_title_font=dict(size=14, weight='bold'),
    yaxis_title_font=dict(size=14, weight='bold'),
    xaxis_tickfont=dict(size=12),
    yaxis_tickfont=dict(size=12),
    xaxis_tickvals=[0, 1],
    xaxis_ticktext=['No', 'Sí'], 
    showlegend=False,
    yaxis=dict(
        title='', 
        tickformat='d' 
    ),
    width=400,  
    height=500 
)

fig.update_traces(
    texttemplate='%{y}', 
    textposition='outside', 
    textfont=dict(size=12, weight='bold', color='white') 
)

fig.show()

##### La distribución de saldo:

In [16]:
fig = px.histogram(
    df, 
    x='balance', 
    nbins=100, 
    title='La distribución de saldo',
    labels={'balance': 'Saldo (balance), euro', 'count': 'Número de clientes'},
    range_x=[-5000, 82000]
)

fig.update_layout(
    title_font=dict(size=16, weight='bold'),
    xaxis_title_font=dict(size=14, weight='bold'),
    yaxis_title_font=dict(size=14, weight='bold'),
    xaxis_tickfont=dict(size=12),
    yaxis_tickfont=dict(size=12),
    yaxis=dict(
        title='Número de clientes', 
        tickformat='d' 
    ),
    xaxis=dict(
        title='Saldo (balance), euro', 
        tickformat='d' 
    ),
    showlegend=False
)

fig.show()

##### Saldo de los clientes sin y con impago:

In [17]:
fig = px.box(
    df_plot, 
    x='default_label', 
    y='balance',
    title='Balance vs Default',
    labels={'default_label': 'Impago (default)', 'balance': 'Euro'},
    category_orders={'default_label': ['No', 'Sí']},
    color='default_label',
    color_discrete_sequence=['#636EFA', '#EF553B']
)

fig.update_layout(
    title_font=dict(size=16, weight='bold'),
    xaxis_title_font=dict(size=14, weight='bold'), 
    yaxis_title_font=dict(size=14, weight='bold'), 
    xaxis_tickfont=dict(size=12), 
    yaxis_tickfont=dict(size=12), 
    yaxis=dict(range=[-8500, 85000], 
               tickformat='d'), 
    showlegend=False
)

fig.show()

##### Indicadores estadísticos del saldo en las cuentas de los clientes sin y con impago:

El conjunto de datos es altamente asimétrico en términos de categoría del balance y número de impagos de préstamos (desequilibrio de clases extremo).

In [18]:
df.groupby('default')['balance'].agg(['mean', 'median', 'min', 'max'])

,mean,median,min,max
default,,,,
0,1562.100129,568.0,-3058,81204
1,-71.440994,0.0,-6847,5249


#### Tasa de impagos en diferentes tipos de empleo (Job vs Default):

In [19]:
job_risk = pd.crosstab(
    df['job'],
    df['default'],
    normalize='index'
)

job_risk

default,0,1
job,,
admin.,0.992413,0.007587
blue-collar,0.979003,0.020997
entrepreneur,0.968750,0.031250
housemaid,0.970037,0.029963
management,0.985404,0.014596
retired,0.993506,0.006494
self-employed,0.982412,0.017588
services,0.992274,0.007726
student,0.997214,0.002786


In [20]:
job_risk_sorted = job_risk.sort_values(by=1, ascending=False)

In [21]:
job_risk_percent = job_risk_sorted * 100

In [22]:
job_risk_percent

default,0,1
job,,
entrepreneur,96.875000,3.125000
housemaid,97.003745,2.996255
blue-collar,97.900262,2.099738
unemployed,97.976879,2.023121
self-employed,98.241206,1.758794
technician,98.438371,1.561629
management,98.540434,1.459566
unknown,98.571429,1.428571
services,99.227373,0.772627


In [23]:
df_plot_job = job_risk_percent.reset_index().melt(id_vars='job', var_name='Impago', value_name='Tasa')

In [24]:
df_plot_job_risk = job_risk_percent.reset_index().melt(id_vars='job', var_name='default', value_name='Porsentage')

In [25]:
df_plot_job_risk

,job,default,Porsentage
0,entrepreneur,0,96.875000
1,housemaid,0,97.003745
2,blue-collar,0,97.900262
3,unemployed,0,97.976879
4,self-employed,0,98.241206
5,technician,0,98.438371
6,management,0,98.540434
7,unknown,0,98.571429
8,services,0,99.227373
9,admin.,0,99.241275


In [26]:
fig = px.bar(
    df_plot_job_risk,
    x='job',
    y='Porsentage',
    color='default',
    color_discrete_sequence=px.colors.qualitative.Set2, 
    title='Tasa de impago por empleo (100% Stacked Bar)',
    labels={'job': '', 'Tasa': 'Tasa de clientes (%)'},
    text='Porsentage', 
    template='custom', 
    barmode='stack'
)


fig.update_layout(
    height=500, 
    legend_title='Impago',
    legend=dict(x=1, y=0, traceorder='normal'), 
    yaxis=dict(range=[0, 110], gridcolor='#444444'), 
    xaxis=dict(tickangle=-45) 
)


fig.update_traces(
    texttemplate='%{y:.1f}%',
    textposition='inside', 
    textfont=dict(size=12, color='white'),
    marker_line_color='black',
    marker_line_width=1,
    selector=dict(name='Sí') 
)

custom_text = []
for index, row in df_plot_job.iterrows():
    if row['Tasa'] >= 5:
        custom_text.append(f"{row['Tasa']:.1f}%")
    else:
        custom_text.append("")

fig.data[0].text = custom_text 
fig.data[1].text = custom_text 


for i, trace in enumerate(fig.data):
    y_vals = trace.y
    new_text = [f"<b>{v:.1f}%</b>" if v >= 5 else "" for v in y_vals]
    trace.text = new_text

fig.show()

#### Tendencias: Edad vs Balance por Estado de Default

In [27]:
fig = px.scatter(
    df,
    x='age',
    y='balance',
      color=df['default'].astype(str),
    opacity=0.5,
    color_discrete_map={'1': '#e74c3c', '0': '#2ecc71'},
    labels={
        'age': "<b>Edad del Cliente</b>", 
        'balance': "<b>Saldo</b>", 
        'color': "<b>Default o no?</b>"  
    },
    title="<b>Análisis de Tendencias: Edad vs Balance por Estado de Default</b>",
)

fig.update_layout(
    width=880,  
    height=480,
    title_font={'size': 16},
    legend_font={'size': 14},
    legend_title_font={'size': 14},
)

fig.update_xaxes(title_font={'size': 14}, tickfont={'size': 12})
fig.update_yaxes(title_font={'size': 14}, tickfont={'size': 12})

fig.show()

#### Tendencias: Edad vs Balance por Estado de Default y por grupos de clientes

Transformar los valores de la variable 'cluster' a los más entendidos

In [28]:
df['cluster'] = df['cluster'].map({0: "0_Jubilado", 1: "1_Management", 2: "2_Technician", 3: "3_Blue-collar"})

In [29]:
df

,id,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,deposit,cluster
0,1,59,admin.,married,secondary,0,2343,1,0,unknown,5,may,1042,1,-1,0,no_pcampaign,1,0_Jubilado
1,2,56,admin.,married,secondary,0,45,0,0,unknown,5,may,1467,1,-1,0,no_pcampaign,1,0_Jubilado
2,3,41,technician,married,secondary,0,1270,1,0,unknown,5,may,1389,1,-1,0,no_pcampaign,1,1_Management
3,4,55,services,married,secondary,0,2476,1,0,unknown,5,may,579,1,-1,0,no_pcampaign,1,0_Jubilado
4,5,54,admin.,married,tertiary,0,184,0,0,unknown,5,may,673,2,-1,0,no_pcampaign,1,0_Jubilado
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10982,10983,40,management,married,secondary,0,8486,0,0,unknown,6,may,260,3,-1,0,no_pcampaign,0,1_Management
10983,10984,53,management,married,tertiary,0,20772,0,0,cellular,4,feb,715,1,-1,0,no_pcampaign,0,0_Jubilado
10984,10985,55,blue-collar,married,primary,0,3297,1,1,telephone,30,apr,96,1,-1,0,no_pcampaign,0,0_Jubilado
10985,10986,41,management,married,tertiary,0,9,1,0,cellular,22,jul,82,3,-1,0,no_pcampaign,0,2_Technician


In [30]:
fig = px.scatter(
    df,
    x='age',
    y='balance',
      color=df['cluster'].astype(str),
    opacity=0.5,
    # color_discrete_map={'1': '#e74c3c', '0': '#2ecc71'},
    labels={
        'age': "<b>Edad del Cliente</b>", 
        'balance': "<b>Saldo</b>", 
        'color': "<b>Grupos de los clientes</b>"  
    },
    title="<b>Análisis de Tendencias: Edad vs Balance por grupos de los clientes</b>",
)

fig.update_layout(
    width=880,  
    height=480,
    title_font={'size': 16},
    legend_font={'size': 14},
    legend_title_font={'size': 14},
)

fig.update_xaxes(title_font={'size': 14}, tickfont={'size': 12})
fig.update_yaxes(title_font={'size': 14}, tickfont={'size': 12})

fig.show()

In [31]:
fig = px.scatter(
    df,
    x='age',
    y='balance',
    color=df['cluster'].astype(str),
    facet_col='default',  
    opacity=0.5,
    labels={
        'age': "", 
        'balance': "<b>Saldo</b>", 
        'color': "<b>Grupos de los clientes</b>"
    },
    title="<b>Análisis de Tendencias: Edad vs Balance por grupos</b>",
)

fig.add_annotation(
    text="<b>Edad del Cliente</b>", 
    xref="paper",
    yref="paper", 
    x=0.5,
    y=-0.15, 
    showarrow=False,
    font=dict(size=14),
    xanchor="center",
    yanchor="top",
)

fig.update_layout(
    width=1600,  
    height=480,
    title_font={'size': 16},
    legend_font={'size': 14},
    legend_title_font={'size': 14},
)

fig.update_xaxes(title_font={'size': 14}, tickfont={'size': 12})
fig.update_yaxes(title_font={'size': 14}, tickfont={'size': 12})

fig.show()

## El analisis

Dividir a los clientes en grupos según el saldo de su cuenta para determinar el riesgo de impago.

Dado que los impagos de préstamos son muy escasos en todos los datos disponibles, para obtener un número suficiente de impagos para el análisis y resultados estadísticos estables a partir del análisis, fue definido a los clientes con saldos bajos como aquellos que se encuentran en el 25% inferior de la distribución de saldos.

##### Determinar el límite superior de saldo bajo (25%):

In [32]:
df

,id,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,deposit,cluster
0,1,59,admin.,married,secondary,0,2343,1,0,unknown,5,may,1042,1,-1,0,no_pcampaign,1,0_Jubilado
1,2,56,admin.,married,secondary,0,45,0,0,unknown,5,may,1467,1,-1,0,no_pcampaign,1,0_Jubilado
2,3,41,technician,married,secondary,0,1270,1,0,unknown,5,may,1389,1,-1,0,no_pcampaign,1,1_Management
3,4,55,services,married,secondary,0,2476,1,0,unknown,5,may,579,1,-1,0,no_pcampaign,1,0_Jubilado
4,5,54,admin.,married,tertiary,0,184,0,0,unknown,5,may,673,2,-1,0,no_pcampaign,1,0_Jubilado
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10982,10983,40,management,married,secondary,0,8486,0,0,unknown,6,may,260,3,-1,0,no_pcampaign,0,1_Management
10983,10984,53,management,married,tertiary,0,20772,0,0,cellular,4,feb,715,1,-1,0,no_pcampaign,0,0_Jubilado
10984,10985,55,blue-collar,married,primary,0,3297,1,1,telephone,30,apr,96,1,-1,0,no_pcampaign,0,0_Jubilado
10985,10986,41,management,married,tertiary,0,9,1,0,cellular,22,jul,82,3,-1,0,no_pcampaign,0,2_Technician


In [33]:
q25 = df['balance'].quantile(0.25)

In [34]:
print(f"El límite superior de saldo bajo es {round(q25)} euro")

El límite superior de saldo bajo es 125 euro


In [35]:
df['low_balance'] = np.where(
    df['balance'] <= q25,
    1,
    0
)

##### Número de clientes en el grupo de saldo bajo y otros clientes:

El número de clientes en el grupo de saldo bajo es 2748.

In [36]:
counts = df['low_balance'].value_counts()
counts

low_balance
0    8239
1    2748
Name: count, dtype: int64

#### Tasa de impago por nivel de balance y grupos de clientes

In [37]:
df_low_balance_cluster = (
    df.groupby(['cluster', 'low_balance', 'default'])
    .size()
    .reset_index(name='count')
)

df_low_balance_cluster['total_in_group'] = df_low_balance_cluster.groupby(['cluster', 'low_balance'])[
    'count'
].transform('sum')

df_low_balance_cluster['porsentage'] = (
    df_low_balance_cluster['count'] / df_low_balance_cluster['total_in_group']
) * 100

df_low_balance_cluster

,cluster,low_balance,default,count,total_in_group,porsentage
0,0_Jubilado,0,0,1820,1829,99.507928
1,0_Jubilado,0,1,9,1829,0.492072
2,0_Jubilado,1,0,475,490,96.938776
3,0_Jubilado,1,1,15,490,3.061224
4,1_Management,0,0,2326,2334,99.657241
5,1_Management,0,1,8,2334,0.342759
6,1_Management,1,0,813,865,93.988439
7,1_Management,1,1,52,865,6.011561
8,2_Technician,0,0,1907,1912,99.738494
9,2_Technician,0,1,5,1912,0.261506


In [38]:
fig = px.bar(
    df_low_balance_cluster,
    x='low_balance', 
    y='porsentage',
    color=df_low_balance_cluster['default'].astype(str),  
    facet_col='cluster', 
    color_discrete_sequence=px.colors.qualitative.Set2, 
    title='<b>Tasa de impago por nivel de balance y grupos de clientes (100% Stacked Bar)</b>',
    labels={
        'low_balance': "", 
        'porsentage': "Tasa de clientes (%)",
        'cluster': "<b>Cluster</b>",
        'color': 'Impago'
    },
    text='porsentage', 
    template='custom', 
    barmode='stack'
)

fig.update_layout(
    height=530, 
    width=880,
    margin=dict(b=80),
    legend_title="Impago",
    legend=dict(x=1, y=0, traceorder='normal'), 
    yaxis=dict(range=[0, 110], gridcolor='#444444'), 
)

fig.update_xaxes(title_text="")
fig.update_xaxes(
    tickvals=[0, 1],
    ticktext=["Normal", "Bajo"],
    tickangle=0 
)

fig.add_annotation(
    text="<b>¿Saldo Bajo?</b>", 
    xref="paper",
    yref="paper", 
    x=0.5,
    y=-0.15, 
    showarrow=False,
    font=dict(size=14),
    xanchor="center",
    yanchor="top",
)

for trace in fig.data:
    if trace.y is not None:
        trace.text = [f"{v:.1f}%" if v >= 5 else "" for v in trace.y]
        trace.texttemplate = '%{text}'
        trace.update(
            textposition='inside', 
            textfont=dict(size=12, color='white'),
            marker_line_color='black',
            marker_line_width=1
        )

fig.show()

##### Proporción de impagos en el grupo de saldo bajo con respecto al resto:

El número de impagos entre el grupo de clientes con saldo bajo es de 132 casos, en comparación con 29 entre los demás clientes.

In [39]:
contingency_balance = pd.crosstab(
    df['low_balance'],
    df['default']
)

contingency_balance

default,0,1
low_balance,,
0,8210,29
1,2616,132


In [40]:
contingency_deposit = pd.crosstab(
    df['deposit'],
    df['default']
)

contingency_deposit

default,0,1
deposit,,
0,5589,109
1,5237,52


#### La probabilidad de incumplimiento

Los clientes con saldos bajos tienen 13,6 veces más probabilidades de incurrir en impago.

In [41]:
default_rates = (
    df.groupby('low_balance')['default']
    .mean()
    .reset_index(name='default_probability')
)

default_rates

,low_balance,default_probability
0,0,0.003520
1,1,0.048035


In [42]:
default_rates_by_cluster = (
    df.groupby(['cluster', 'low_balance'])['default']
    .mean()
    .reset_index(name='default_probability')
)

default_rates_by_cluster['default_percentage'] = default_rates_by_cluster['default_probability'] * 100

default_rates_by_cluster

,cluster,low_balance,default_probability,default_percentage
0,0_Jubilado,0,0.004921,0.492072
1,0_Jubilado,1,0.030612,3.061224
2,1_Management,0,0.003428,0.342759
3,1_Management,1,0.060116,6.011561
4,2_Technician,0,0.002615,0.261506
5,2_Technician,1,0.043873,4.387292
6,3_Blue-collar,0,0.003235,0.323475
7,3_Blue-collar,1,0.049180,4.918033


##### Riesgo relativo (RR) para todos clientes:

Los clientes con saldos bajos incurren en impago 13,65 veces más frequente.

In [43]:
low_risk = default_rates.loc[
    default_rates['low_balance'] == 1,
    'default_probability'
].values[0]

normal_risk = default_rates.loc[
    default_rates['low_balance'] == 0,
    'default_probability'
].values[0]

relative_risk = low_risk / normal_risk

print(round(relative_risk,2))

13.65


In [44]:
fig = px.bar(
    default_rates, 
    x='low_balance', 
    y='default_probability',
    color='default_probability',
    title='La probabilidad de impago entre los grupos por saldo',
    labels={'low_balance': '', 'default_probability': 'Probabilidad de impago'},
    category_orders={'low_balance': [0, 1]},
    color_continuous_scale=['green', 'red'] 
)

fig.update_layout(
    title_font=dict(size=16, weight='bold'),
    xaxis_title_font=dict(size=14, weight='bold'),
    yaxis_title_font=dict(size=14, weight='bold'),
    xaxis_tickfont=dict(size=12),
    yaxis_tickfont=dict(size=12),
    xaxis=dict(
        tickvals=[0, 1],
        ticktext=['Saldo normal', 'Saldo bajo'] 
    ),
    yaxis=dict(
        title="Probabilidad de impago",
        tickformat='.0%' 
    ),
    showlegend=False,
    width=600,  
    height=500 
)

fig.show()

##### Riesgo relativo (RR) para los clientes por grupos:

In [45]:
default_rates_by_cluster = (
    df.groupby(['cluster', 'low_balance'])['default']
    .mean()
    .reset_index(name='probability')
)


df_pivot = default_rates_by_cluster.pivot(
    index='cluster', columns='low_balance', values='probability'
).reset_index()


df_pivot = df_pivot.rename(columns={1: "probability_low", 0: "probability_normal"})
df_pivot['relative_risk'] = df_pivot['probability_low'] / df_pivot['probability_normal']


relative_risk_by_cluster = pd.DataFrame(
    {
        'cluster': df_pivot['cluster'],
        'probability_low_balance (%)': (df_pivot['probability_low'] * 100).round(2),
        'probability_normal_balance (%)': (df_pivot['probability_normal'] * 100).round(2),
        'relative_risk (RR)': (df_pivot['relative_risk']).round(2),
    }
)

relative_risk_by_cluster

,cluster,probability_low_balance (%),probability_normal_balance (%),relative_risk (RR)
0,0_Jubilado,3.06,0.49,6.22
1,1_Management,6.01,0.34,17.54
2,2_Technician,4.39,0.26,16.78
3,3_Blue-collar,4.92,0.32,15.20


In [46]:
default_rates_by_cluster['probability_%'] = default_rates_by_cluster['probability'] * 100

In [47]:
default_rates_by_cluster

,cluster,low_balance,probability,probability_%
0,0_Jubilado,0,0.004921,0.492072
1,0_Jubilado,1,0.030612,3.061224
2,1_Management,0,0.003428,0.342759
3,1_Management,1,0.060116,6.011561
4,2_Technician,0,0.002615,0.261506
5,2_Technician,1,0.043873,4.387292
6,3_Blue-collar,0,0.003235,0.323475
7,3_Blue-collar,1,0.049180,4.918033


In [48]:
fig = px.bar(
    default_rates_by_cluster,
    x='cluster',
    y='probability_%',
    barmode="group", 
    text='probability_%',
    color=default_rates_by_cluster['low_balance'].astype(str),
    color_discrete_map={
        '1': '#e74c3c',
        '0': '#2ecc71',
    },
    title="<b>Riesgos de impago por grupos de clientes y nivel de saldo</b>",
    labels={
        'cluster': "<b>Grupos de clientes</b>",
        'probability_%': "<b>Tasa de Impago (%)</b>",
        'low_balance': "<b>Nivel de Balance</b>",
    },
    template="custom"
)

fig.update_traces(
    texttemplate="%{text:.2f}%",
    textposition='outside',
    textfont=dict(size=12, weight='bold'),
    marker_line_color='black',
    marker_line_width=0.5,
)

fig.update_layout(
    height=500,
    width=800,
    legend_title="Nivel de Balance",
    legend=dict(x=1.02, y=0.5, traceorder='normal'),
    legend_font={'size': 12},
    yaxis=dict(
        gridcolor='#444444',
        range=[0, default_rates_by_cluster['probability_%'].max() * 1.2],
    ),
    xaxis=dict(
        type='category',
        title=dict(standoff=25)
    ), 
)

fig.for_each_trace(lambda t: t.update(name="Bajo" if t.name == '1' else "Normal"))

fig.show()

#### Crear 4 grupos por los tipos de créditos

Dividir a los clientes en 4 grupos por los tipos de créditos los que tienen

Crear una funcion y aplicarla para marcar a los clientes

In [49]:
def credit_category(row):
    if row['housing'] == 1 and row['loan'] == 1:
        return 'loan+housing'
    elif row['housing'] == 1:
        return 'housing'
    elif row['loan'] == 1:
        return 'loan'
    else:
        return 'no credit'

In [50]:
df['credit'] = df.apply(credit_category, axis=1)

In [51]:
df.head()

,id,age,job,marital,education,default,balance,housing,loan,contact,...,month,duration,campaign,pdays,previous,poutcome,deposit,cluster,low_balance,credit
0,1,59,admin.,married,secondary,0,2343,1,0,unknown,...,may,1042,1,-1,0,no_pcampaign,1,0_Jubilado,0,housing
1,2,56,admin.,married,secondary,0,45,0,0,unknown,...,may,1467,1,-1,0,no_pcampaign,1,0_Jubilado,1,no credit
2,3,41,technician,married,secondary,0,1270,1,0,unknown,...,may,1389,1,-1,0,no_pcampaign,1,1_Management,0,housing
3,4,55,services,married,secondary,0,2476,1,0,unknown,...,may,579,1,-1,0,no_pcampaign,1,0_Jubilado,0,housing
4,5,54,admin.,married,tertiary,0,184,0,0,unknown,...,may,673,2,-1,0,no_pcampaign,1,0_Jubilado,0,no credit


Número de impagos en cada grupo de los clientes por tipos de préstamo:

In [52]:
pd.crosstab(
    df['credit'],
    df['default']
)

default,0,1
credit,,
housing,4304,61
loan,580,33
loan+housing,796,22
no credit,5146,45


#### Crear las tablas de medidas

Formar un marcador que muestra el tipo de préstamo y el nivel de saldo para dividir a los clientes por 8 subgrupos: 

In [53]:
df['credit_balance'] = (
    df['credit'] + " - Balance: " + df['low_balance'].astype(str)
)

Calcular la tabla de las medidas claves por 8 subgrupos de clientes:

In [54]:
df_stats = (
    df.groupby(['credit', 'low_balance', 'credit_balance'])
    .agg(
        total_clients=('default', 'count'),
        defaults=('default', lambda x: (x == 1).sum()),
        default_rate=('default', lambda x: (x == 1).mean()),
    )
    .reset_index()
)

In [55]:
df_stats['non_defaults'] = df_stats['total_clients'] - df_stats['defaults']

In [56]:
df_stats

,credit,low_balance,credit_balance,total_clients,defaults,default_rate,non_defaults
0,housing,0,housing - Balance: 0,3205,12,0.003744,3193
1,housing,1,housing - Balance: 1,1160,49,0.042241,1111
2,loan,0,loan - Balance: 0,385,6,0.015584,379
3,loan,1,loan - Balance: 1,228,27,0.118421,201
4,loan+housing,0,loan+housing - Balance: 0,521,3,0.005758,518
5,loan+housing,1,loan+housing - Balance: 1,297,19,0.063973,278
6,no credit,0,no credit - Balance: 0,4128,8,0.001938,4120
7,no credit,1,no credit - Balance: 1,1063,37,0.034807,1026


Como se puede observar en el gráfico "La probabilidad de impago entre los grupos por saldo", el saldo bajo es un predictor de impago muy fuerte, por eso podemos considerar que los clientes sin préstamos actuales y con saldo normal (no bajo) están en el grupo básico con el que se puede comparar a los demás.

Formar un grupo básico:

In [57]:
base_condition = (df_stats['credit'] == 'no credit') & (
    df_stats['low_balance'] == 0
)
base_group = df_stats[base_condition].iloc[0]

In [58]:
base_group

credit                         no credit
low_balance                            0
credit_balance    no credit - Balance: 0
total_clients                       4128
defaults                               8
default_rate                    0.001938
non_defaults                        4120
Name: 6, dtype: object

Dividir el grupo básico por con o sin impago (default)

In [59]:
base_def = base_group['defaults']
base_non_def = base_group['non_defaults']

La prueba del significado estadístico. Como los casos de impago son muy pocos en todo dataset y particularmente en cada de 4 grupos, se puede hacer la prueba exacta de Fisher en lugar de la prueba de chi-cuadrado. La prueba se hace para cada par "default / no default" en cada grupo de tipo préstamo.

Fisher Exact Test:

In [60]:
fisher_results = []

for idx, row in df_stats.iterrows():
    if row['credit'] == 'no credit' and row['low_balance'] == 0:
        fisher_results.append(
            {
                'credit_balance': row['credit_balance'],
                'odds_ratio': 1.0,
                'p_value': 1.0,
                'significance': False,
            }
        )
        continue

    contingency_matrix = [[row['defaults'], row['non_defaults']], [base_def, base_non_def]]

    odds_ratio, p_value = stats.fisher_exact(contingency_matrix)

    fisher_results.append(
        {
            'credit_balance': row['credit_balance'],
            'odds_ratio': odds_ratio,
            'p_value': p_value,
            'significance': p_value < 0.05,
        }
    )

In [61]:
df_fisher = pd.DataFrame(fisher_results)

In [62]:
df_fisher

,credit_balance,odds_ratio,p_value,significance
0,housing - Balance: 0,1.935484,1.761308e-01,False
1,housing - Balance: 1,22.713771,5.969470e-25,True
2,loan - Balance: 0,8.153034,6.158215e-04,True
3,loan - Balance: 1,69.179104,9.126794e-29,True
4,loan+housing - Balance: 0,2.982625,1.167139e-01,False
5,loan+housing - Balance: 1,35.197842,4.008534e-17,True
6,no credit - Balance: 0,1.000000,1.000000e+00,False
7,no credit - Balance: 1,18.572125,7.637376e-19,True


Formar un dataframe unido para visualización:

In [63]:
df_final = pd.merge(df_stats, df_fisher, on='credit_balance')

Ordenar por valores de tipo de préstamo:

In [64]:
df_final = df_final.sort_values(by=['credit', 'low_balance'], ascending=[True, True])

In [65]:
df_final

,credit,low_balance,credit_balance,total_clients,defaults,default_rate,non_defaults,odds_ratio,p_value,significance
0,housing,0,housing - Balance: 0,3205,12,0.003744,3193,1.935484,1.761308e-01,False
1,housing,1,housing - Balance: 1,1160,49,0.042241,1111,22.713771,5.969470e-25,True
2,loan,0,loan - Balance: 0,385,6,0.015584,379,8.153034,6.158215e-04,True
3,loan,1,loan - Balance: 1,228,27,0.118421,201,69.179104,9.126794e-29,True
4,loan+housing,0,loan+housing - Balance: 0,521,3,0.005758,518,2.982625,1.167139e-01,False
5,loan+housing,1,loan+housing - Balance: 1,297,19,0.063973,278,35.197842,4.008534e-17,True
6,no credit,0,no credit - Balance: 0,4128,8,0.001938,4120,1.000000,1.000000e+00,False
7,no credit,1,no credit - Balance: 1,1063,37,0.034807,1026,18.572125,7.637376e-19,True


#### La probabilidad de incumplimiento por los grupos de clientes (clusters)

In [66]:
default_rates_cluster_credit = (
    df.groupby(['cluster', 'credit', 'low_balance'])['default']
    .mean()
    .reset_index(name='default_probability')
)

default_rates_cluster_credit['default_percentage'] = (
    default_rates_cluster_credit["default_probability"] * 100
    ).round(2)


default_rates_cluster_credit

,cluster,credit,low_balance,default_probability,default_percentage
0,0_Jubilado,housing,0,0.011792,1.18
1,0_Jubilado,housing,1,0.007634,0.76
2,0_Jubilado,loan,0,0.020202,2.02
3,0_Jubilado,loan,1,0.078947,7.89
4,0_Jubilado,loan+housing,0,0.000000,0.00
5,0_Jubilado,loan+housing,1,0.026316,2.63
6,0_Jubilado,no credit,0,0.001630,0.16
7,0_Jubilado,no credit,1,0.035336,3.53
8,1_Management,housing,0,0.001689,0.17
9,1_Management,housing,1,0.045131,4.51


In [67]:
contingency_cluster_credit_balance = (
    df.groupby(['cluster', 'credit', 'low_balance', 'default'])
    .size()
    .unstack(fill_value=0)
)

Fisher Exact Test:

In [68]:
contingency_cluster_credit_balance = (
    df.groupby(['cluster', 'credit', 'low_balance', 'default'])
    .size()
    .unstack(fill_value=0)
)

contingency_cluster_credit_balance.columns = ["no_default", "default"]
contingency_cluster_credit_balance = contingency_cluster_credit_balance.reset_index()


fisher_results_cluster = []

for (cluster, credit), group in contingency_cluster_credit_balance.groupby(['cluster', 'credit']):
    low_row = group[group['low_balance']==1]
    normal_row = group[group['low_balance']==0]

    if low_row.empty or normal_row.empty:
        p_val = np.nan
    else:
        table = [
            [low_row["no_default"].values[0], low_row["default"].values[0]],
            [normal_row["no_default"].values[0], normal_row["default"].values[0]],
        ]

        odds_ratio, p_val = stats.fisher_exact(table)

    fisher_results_cluster.append(
        {'cluster': cluster, 'credit': credit, 'odds_ratio': odds_ratio, 'p_value': p_val}
    )


df_fisher_cluster = pd.DataFrame(fisher_results_cluster)


risk_cluster_pivoted = default_rates_cluster_credit.pivot(
    index=['cluster', 'credit'],
    columns='low_balance',
    values='default_percentage',
).reset_index()

df_final_cluster_credit = pd.merge(risk_cluster_pivoted, df_fisher_cluster, on=['cluster', 'credit'])


df_final_cluster_credit["significant (p < 0.05)"] = df_final_cluster_credit["p_value"].apply(
    lambda x: "Yes" if x < 0.05 else "No"
)

df_final_cluster_credit["p_value"] = df_final_cluster_credit["p_value"].round(5)

df_final_cluster_credit.sort_values(
    by=["significant (p < 0.05)", "p_value"], ascending=[False, True]
    )

,cluster,credit,0,1,odds_ratio,p_value,significant (p < 0.05)
3,0_Jubilado,no credit,0.16,3.53,0.044571,0.00000,Yes
4,1_Management,housing,0.17,4.51,0.035800,0.00000,Yes
11,2_Technician,no credit,0.00,2.81,0.000000,0.00000,Yes
12,3_Blue-collar,housing,0.44,5.81,0.071920,0.00000,Yes
7,1_Management,no credit,0.49,4.96,0.094533,0.00001,Yes
8,2_Technician,housing,0.14,3.41,0.041003,0.00006,Yes
5,1_Management,loan,0.83,14.10,0.051184,0.00018,Yes
6,1_Management,loan+housing,0.47,8.06,0.053271,0.00025,Yes
15,3_Blue-collar,no credit,0.19,2.77,0.068106,0.00026,Yes
13,3_Blue-collar,loan,0.00,9.38,0.000000,0.00316,Yes


**Conclusión**:

Dividir a los clientes en grupos y analizar la probabilidad de impago según el tipo de préstamo y el saldo no tiene sentido, ya que los grupos resultantes no contienen suficientes observaciones para extraer conclusiones estadísticamente significativas.

#### Visualización

Visualización de todos valores de probabilidad de impago por cada subgrupo y el significado estadístico:

In [69]:
custom_labels = {
    0: "Normal",
    1: "Bajo"
}


fig = make_subplots(
    rows=1, cols=2, 
    shared_yaxes=True,
    horizontal_spacing=0.05,
    subplot_titles=(
        "<b>Tasa de impagos (Default Rate)<br>por tipos de préstamos y nivel de saldo</b>",
        "<b>El cálculo de riesgo (Odds Ratio)<br>respecto a 'no hay préstamos' y saldo normal</b>"
    )
)

categories = sorted(df_final['low_balance'].unique())
colors = ['#2ecc71', '#CD5C5C'] 
y_order = df_final['credit'].unique()


for idx, cat in enumerate(categories):
    df_sub = df_final[df_final['low_balance'] == cat]
    
    label_text = custom_labels.get(cat, f"Saldo bajo: {cat}")
    
    fig.add_trace(
        go.Bar(
            x=df_sub['default_rate'],
            y=df_sub['credit'],
            name=label_text, 
            orientation='h',
            marker_color=colors[idx],
            legendgroup=str(cat),
            showlegend=True
        ),
        row=1, col=1
    )
    

    fig.add_trace(
        go.Bar(
            x=df_sub['odds_ratio'],
            y=df_sub['credit'],
            orientation='h',
            marker_color=colors[idx],
            opacity=0.6,
            legendgroup=str(cat),
            showlegend=False 
        ),
        row=1, col=2
    )


fig.add_vline(
    x=1, 
    line_width=1.5, 
    line_dash='dash', 
    line_color='white', 
    row=1, col=2
)


num_categories = len(categories)
bar_gap = 0.2 
group_width = 1.0 - bar_gap
bar_width = group_width / num_categories

for row in df_final.itertuples():
    p_text = f"p={row.p_value:.3f}" if row.p_value >= 0.001 else "p<0.001"
    if row.credit == 'no credit' and row.low_balance == 0:
        p_text = "BASE"
        
    marker = " ★" if row.significance else ""
    full_text = f" <b>{p_text}{marker}</b>"
    text_color = '#e74c3c' if row.significance else 'gray'
    
    y_idx = list(y_order).index(row.credit)
    cat_idx = categories.index(row.low_balance)
    y_offset = (cat_idx - (num_categories - 1) / 2) * bar_width
    
    fig.add_annotation(
        x=row.odds_ratio + 0.05,
        y=y_idx + y_offset,
        text=full_text,
        showarrow=False,
        xanchor='left',
        yanchor='middle',
        font={'size': 12, 'color': text_color},
        xref="x2",
        yref="y2"
    )


fig.update_layout(
    width=1280, 
    height=560,
    barmode='group',
    legend_title_text="<b>Saldo:</b>",
    legend_font={'size': 14},
    legend_title_font={'size': 14},
)


for annotation in fig['layout']['annotations']:
    if 'text' in annotation and "<br>" in annotation['text']:  
        annotation['font'] = {'size': 16}


fig.update_xaxes(title_text="<b>Tasa de clientes morosos</b>", title_font={'size': 14}, tickfont={'size': 14}, gridcolor="rgba(255, 255, 255, 0.1)", row=1, col=1)
fig.update_xaxes(title_text="<b>Odds Ratio (Cuántas veces mayor es el riesgo)</b>", title_font={'size': 14}, tickfont={'size': 14}, gridcolor="rgba(255, 255, 255, 0.1)", row=1, col=2)
fig.update_yaxes(title_text="<b>Tipo de préstamo y nivel de saldo</b>", title_font={'size': 14}, tickfont={'size': 14}, categoryorder='array', categoryarray=y_order, row=1, col=1)

fig.show()

## RESUMEN

#### La probabilidad de incumplimiento: 

- SI, los clientes con saldos bajos tienen más probabilidades de incurrir en impago con todos tipos de préstamos, que significa que el saldo menos de 125 euro es un predictor fuerte de impago.

- Las personas con préstamo individual ("loan") tienen el riesgo de incumplimiento mas de 69,18 veces y probabilidad de impago casi 12% sí además su saldo es bajo, y solo 8,15 veces si su saldo es normal.

- El segundo puesto en esta clasificación negativa es para "loan+housing" - 35,20 veces con saldo bajo. 

- El préstamo hipotecário ("housing") está en el tercer puesto - 22,71 y 4% de probabilidad de impago.

- Los resultados son significntes estadísticamente según la prueba exacta de Fisher (Fisher Exact Test).

- No es posible confirmar los cálculos del riesgo y la probabilidad de impago para las personas que tienen los préstamos "loan+housing" y "housing" y además saldo normal, porque los casos como así son muy pocos y no tienen significado estadístico.

#### Recomendaciones para la política de riesgos

Respecto a los clientes clasificados con un alto riesgo de default (especialmente aquellos con saldos bajos), se recomienda endurecer la política crediticia mediante las siguientes acciones:

   - Implementar un sistema de seguimiento del saldo del cliente.

   - Implementar un sistema de los coeficientes que valoran las características sociodemográficos del cliente: estado civil, puesto de trabajo.

   - Reducir el límite de crédito disponible a los clientes que tienen la probabiliad de impago alta.

   - Incrementar la tasa de interés aplicable.

   - Disminuir el plazo de amortización del financiamiento.